# Study: 200 CSV, 12 cases, 80 iterations

The previous run was stopped and all generated results were deleted at the user request. Input datasets and configuration are preserved. Run cells remain disabled by default. A future study starts from scratch.


In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/notebook_study.json").exists())
sys.path.insert(0, str(ROOT / "implementation"))
from src.notebook_study import cases, plan, run_study, read_results, build_tables, GROUPS, read_task, task_paths, task_filename
CONFIG = json.loads((ROOT / "configs/notebook_study.json").read_text(encoding="utf-8"))
OUTPUT = ROOT / CONFIG["output"]
display(pd.DataFrame([CONFIG]))

## 1. Thiết kế và khối lượng

200 CSV × 12 case × 10 seed = 24.000 lượt tối ưu. Mỗi lượt 80 vòng, quần thể 25, 4 UAV. Quần thể nền dùng chung trên cùng dataset/seed; K-Means thay một cá thể có chủ đích.

In [ ]:
design = pd.DataFrame(cases())
for group, names in GROUPS.items():
    display(Markdown(f"### {group}"))
    display(design.set_index("case").loc[names].reset_index().fillna("?"))
files, variants, total = plan(ROOT, CONFIG)
print(f"{len(files)} dataset; {len(variants)} cases; {len(CONFIG['seeds'])} seed; {total} tasks")

## 2. Ô chạy — mặc định tắt

Chỉ bật khi muốn khởi chạy/tiếp tục và không có coordinator khác. Luồng thực thi dùng `run_notebook_study.py --workers 4`, có checkpoint theo task. Không sửa source/config/data giữa chừng một lần chạy.

In [ ]:
CONFIRM_RUN = False
if CONFIRM_RUN:
    import subprocess
    subprocess.run([sys.executable, "-B", str(ROOT / "implementation/scripts/run_notebook_study.py"), "--workers", "4"], cwd=ROOT, check=True)
else:
    print("Run disabled. The approved study has 30 seeds and 450000 tasks. Do not start a second coordinator.")


## 3. Bảng từ kết quả thật

Chờ completion.json trước khi tổng hợp. Các ô dưới đọc lại artifact; không gọi optimizer. SD qua seed được tính riêng từng dataset trước khi gộp. FE bao gồm ứng viên bị loại; không tính lại metric báo cáo.

In [ ]:
tables = None
if (OUTPUT / "completion.json").exists():
    saved = sorted((OUTPUT / "tables").glob("*.csv"))
    tables = {p.stem: pd.read_csv(p) for p in saved} if saved else build_tables(OUTPUT)
    for name, table in tables.items():
        display(Markdown(f"### {name} ({len(table)} rows; first 100 shown)"))
        display(table.head(100))
else:
    progress = OUTPUT / "progress.json"
    if progress.exists():
        display(pd.DataFrame([json.loads(progress.read_text(encoding="utf-8"))]))
    print("Study running or not started; final tables will appear after completion.")


## 4. Hội tụ

Đường trung vị và dải phân vị 25–75% trên dataset/seed. IQR không phải khoảng tin cậy. Vòng 0 là sau khởi tạo; mốc 35/80 thuộc lần chạy 80 vòng.

In [ ]:
if tables is not None:
    from src.study_report import build_figures
    from IPython.display import Image
    images = build_figures(ROOT, OUTPUT, tables)
    for path in images:
        if path.name.startswith("convergence_"):
            display(Image(filename=str(path)))


## 5. Kết quả theo quy mô

Mỗi điểm tổng hợp các dataset/seed cùng số user. Hiển thị chất lượng, các proxy, thời gian quan sát và số FE. Không cố định trục phủ sóng ở sát 100%.

In [ ]:
if tables is not None:
    from IPython.display import Image
    for path in sorted((OUTPUT / "figures").glob("scale_*.png")):
        display(Image(filename=str(path)))


## 6. Minh họa 3D

Chọn dataset 200 user đầu tiên theo tên, seed nhỏ nhất, I-WOA ở vòng 80. Đọc nghiệm đã lưu; không tối ưu lại hoặc chọn hình theo thắng/thua.

In [ ]:
if tables is not None:
    from IPython.display import Image
    display(Image(filename=str(OUTPUT / "figures/placement_3d.png")))


## 7. Nhận xét

Đọc ranking, contributions và stability; báo cả chất lượng, chi phí và ổn định. Bảng hiện là mô tả, chưa có kiểm định/CI cho 12 trường hợp. Cùng 80 vòng không đồng nghĩa cùng FE. Chỉ kết luận sau khi có đầy đủ dữ liệu thực.